# WC 2026 Match Probabilities: Multi-Model Comparison

Models: `gemini_pro_31`, `gpt_52`, `grok_42`, `opus_46`, `sonnet_46`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DATA_DIR = '../data/'

MODEL_FILES = {
    'Gemini Pro 3.1': DATA_DIR + 'wc_2026_match_probabilities_gemini_pro_31.csv',
    'GPT-5s':         DATA_DIR + 'wc_2026_match_probabilities_gpt_52.csv',
    'Grok 4.2':       DATA_DIR + 'wc_2026_match_probabilities_grok_42.csv',
    'Opus 4.6':       DATA_DIR + 'wc_2026_match_probabilities_opus_46.csv',
    'Sonnet 4.6':     DATA_DIR + 'wc_2026_match_probabilities_sonnet_46.csv',
}
AVG_OUT = DATA_DIR + 'wc_2026_match_probabilities_avg.csv'
N_OPPONENTS = 47

MODEL_COLORS = {
    'Gemini Pro 3.1': '#4285F4',
    'GPT-5s':         '#10A37F',
    'Grok 4.2':       '#1DA1F2',
    'Opus 4.6':       '#C97B3A',
    'Sonnet 4.6':     '#A855F7',
}

## Step 1 — Build average LLM probability CSV

In [ ]:
def normalize(df):
    """Ensure team_1 < team_2 alphabetically so joins align correctly."""
    mask = df['team_1'] > df['team_2']
    flipped = df[mask].rename(columns={
        'team_1': 'team_2', 'team_2': 'team_1',
        'team_1_win_prob': 'team_2_win_prob',
        'team_2_win_prob': 'team_1_win_prob',
    })[df.columns]
    return (
        pd.concat([df[~mask], flipped])
        .sort_values(['team_1', 'team_2'])
        .reset_index(drop=True)
    )

def team_scores(df, n=N_OPPONENTS):
    """Average win probability across all 47 opponents."""
    as_t1 = df.groupby('team_1')['team_1_win_prob'].sum()
    as_t2 = df.groupby('team_2')['team_2_win_prob'].sum()
    return ((as_t1.add(as_t2, fill_value=0)) / n).sort_values(ascending=False)

raw    = {name: pd.read_csv(path) for name, path in MODEL_FILES.items()}
normed = {name: normalize(df).set_index(['team_1', 'team_2']) for name, df in raw.items()}

# Equal-weight average across all 5 models
avg_df = (
    pd.concat(normed.values())
    .groupby(level=[0, 1])
    .mean()
    .round(4)
    .reset_index()
)

row_sums = avg_df[['team_1_win_prob', 'team_2_win_prob', 'draw_prob']].sum(axis=1)
print(f'Average CSV rows : {len(avg_df)}')
print(f'Row sum range    : [{row_sums.min():.4f}, {row_sums.max():.4f}]')

avg_df.to_csv(AVG_OUT, index=False)
print(f'Saved: {AVG_OUT}')

# Compute team scores for every model + the average
scores     = {name: team_scores(df) for name, df in raw.items()}
avg_scores = team_scores(avg_df)

print('\nConsensus top 10:')
display(avg_scores.head(10).rename('avg_win_prob').to_frame())

## Step 2 — Per-model analysis

In [ ]:
for name in MODEL_FILES:
    model_scores = scores[name]
    color        = MODEL_COLORS[name]

    # Raw delta vs consensus
    raw_delta = (model_scores - avg_scores).dropna()

    # Center the delta to remove overall calibration bias.
    # e.g. Grok compresses all win probs down (high draw rates) so every team
    # is negative vs consensus — centering isolates *relative* preferences.
    delta = (raw_delta - raw_delta.mean()).sort_values(ascending=False)

    top10  = model_scores.head(10)
    loved  = delta.head(5)
    dreads = delta.tail(5).sort_values()

    # ── Figure ────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(name, fontsize=16, fontweight='bold', color=color, x=0.02, ha='left')

    # Left: top 10 favorites
    ax = axes[0]
    bars = ax.barh(top10.index[::-1], top10.values[::-1], color=color, alpha=0.85)
    ax.set_title('Top 10 Favorites (avg win prob)', fontsize=11)
    ax.set_xlabel('Avg win probability')
    ax.set_xlim(0, top10.max() * 1.18)
    for bar, val in zip(bars, top10.values[::-1]):
        ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height() / 2,
                f'{val:.3f}', va='center', fontsize=9)

    # Middle: loved (model over-indexes vs consensus, after centering)
    ax = axes[1]
    bars = ax.barh(loved.index[::-1], loved.values[::-1], color=color, alpha=0.85)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title('Loved by this model\n(centered Δ vs consensus, top 5)', fontsize=11)
    ax.set_xlabel('Centered Δ avg win prob')
    for bar, val in zip(bars, loved.values[::-1]):
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
                f'+{val:.3f}', va='center', fontsize=9)

    # Right: dreaded (model under-indexes vs consensus, after centering)
    ax = axes[2]
    bars = ax.barh(dreads.index, dreads.values, color='#E05C5C', alpha=0.85)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title('Dreaded by this model\n(centered Δ vs consensus, bottom 5)', fontsize=11)
    ax.set_xlabel('Centered Δ avg win prob')
    for bar, val in zip(bars, dreads.values):
        ax.text(bar.get_width() - 0.001, bar.get_y() + bar.get_height() / 2,
                f'{val:+.3f}', va='center', ha='right', fontsize=9, color='white')

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()